# Setup

## Import dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import src.datasets as datasets
import src.LieTorch as lt
import src.LieTorch.nn.functional as F

## Set up experiment parameters

In [ ]:
train_fraction = 0.8
k_folds = 5

seed = None
rng = np.random.default_rng(seed)

## Load dataset

### Load full dataset

In [ ]:
dataset, metadata = datasets.load_breast_cancer_dataset()
num_instances_dataset = metadata['num_instances']
num_features = metadata['num_features']
num_classes = 2

### Split test and train sets

In [ ]:
num_instances_train = int(train_fraction * num_instances_dataset)
num_instances_test = num_instances_dataset - num_instances_train

# TODO: stratify the split
dataset_train, dataset_test = datasets.random_split(dataset, [num_instances_train, num_instances_test], generator=rng)

## Create Model

In [ ]:
def create_model(input_dim, output_dim=2, dropout_p=0.2, hidden_dimension=128, hidden_layers=2):
    model = lt.nn.Sequential()
    prev_layer_output_dim = input_dim
    for i in range(hidden_layers):
        model.add_module(f'linear{i+1}', lt.nn.Linear(in_features=prev_layer_output_dim, out_features=hidden_dimension, init='he_normal'))
        model.add_module(f'relu{i+1}', lt.nn.ReLU())
        model.add_module(f'dropout{i+1}', lt.nn.Dropout(p=dropout_p))
        prev_layer_output_dim = hidden_dimension
    model.add_module('linear_out', lt.nn.Linear(in_features=prev_layer_output_dim, out_features=output_dim, init='he_normal'))
    return model

# Model training

## Train

In [ ]:
def train_one_epoch(model, dataset_iterator, optim):
    model.train()
    epoch_loss = 0.0
    for x, y in dataset_iterator:
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        epoch_loss += loss.item()
        loss.backward()
        optim.step()
        optim.zero_grad()
    return epoch_loss / len(dataset_iterator)


## Test

In [ ]:
def accuracy_from_cm(confusion_matrix):
    return (confusion_matrix[0, 0] + confusion_matrix[1, 1]) / np.sum(confusion_matrix)

def f1_from_cm(confusion_matrix):
    tp = confusion_matrix[1, 1]
    fp = confusion_matrix[1, 0]
    fn = confusion_matrix[0, 1]
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

def test(model, dataset_iterator):
    model.eval()
    cm = np.zeros((2, 2), dtype=int)  # Confusion matrix: [[TN, FP], [FN, TP]]
    for x, y in dataset_iterator:
        with lt.no_grad():
            logits = model(x)
        predicted_labels = np.argmax(logits.numpy(), axis=1)
        cm[1, 1] += np.sum((predicted_labels == 1) & (y._data == 1))  # TP
        cm[0, 0] += np.sum((predicted_labels == 0) & (y._data == 0))  # TN
        cm[1, 0] += np.sum((predicted_labels == 1) & (y._data == 0))  # FP
        cm[0, 1] += np.sum((predicted_labels == 0) & (y._data == 1))  # FN

    return cm

## Training loop

In [ ]:
def train_model(model, train_iterator, test_iterator, optim, epochs):
    train_loss_history = []
    train_confusion_history = []
    test_confusion_history = []
    acc_train_history = []
    f1_train_history = []
    acc_test_history = []
    f1_test_history = []
    for epoch in range(epochs):
        loss = train_one_epoch(model, train_iterator, optim)
        confusion_train = test(model, train_iterator)
        confusion_test = test(model, test_iterator)
        train_confusion_history.append(confusion_train)
        test_confusion_history.append(confusion_test)

        train_loss_history.append(loss)
        acc_train_history.append(accuracy_from_cm(confusion_train))
        f1_train_history.append(f1_from_cm(confusion_train))
        acc_test_history.append(accuracy_from_cm(confusion_test))
        f1_test_history.append(f1_from_cm(confusion_test))
    
    return train_loss_history, train_confusion_history, test_confusion_history

## TODO: Hyperparameter Tuning with K-Fold Cross Validation

In [ ]:
# hardcoded hyperparameters for now, but we can later add a grid/random search or bayesian optimization later
batch_size = 32
hidden_dimension = 20
dropout_p = 0.2
epochs = 500
learning_rate = 1e-4

In [ ]:

train_mu, train_sigma = dataset_train.X.mean(), dataset_train.X.std()
std_dataset_train = datasets.normalize_features(dataset_train, train_mu, train_sigma)
std_dataset_test = datasets.normalize_features(dataset_test, train_mu, train_sigma)

train_iterator = datasets.DataLoader(std_dataset_train, batch_size=batch_size, shuffle=True, rng=rng)
test_iterator = datasets.DataLoader(std_dataset_test, batch_size=batch_size, shuffle=False, rng=rng)

model = create_model(num_features, num_classes, dropout_p, hidden_dimension)
optim = lt.optim.SGD(model.parameters(), lr=learning_rate)
train_loss_history, train_confusion_history, test_confusion_history = train_model(model, train_iterator, test_iterator, optim, epochs)

# Performance

## Plot loss

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(train_loss_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Avg Loss')
plt.legend()
plt.show()

## Plot accuracy

In [ ]:
plt.figure(figsize=(5, 3))
plt.ylim(0, 1)
plt.plot([accuracy_from_cm(cm) for cm in train_confusion_history], label='Train Accuracy')
plt.plot([accuracy_from_cm(cm) for cm in test_confusion_history], label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Plot F1

In [ ]:
plt.figure(figsize=(5, 3))
plt.ylim(0, 1)
plt.plot([f1_from_cm(cm) for cm in train_confusion_history], label='Train F1 Score')
plt.plot([f1_from_cm(cm) for cm in test_confusion_history], label='Test F1 Score')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.show()

# Interpretability

## Feature Vectors (2D reduction)

In [ ]:
def plot_features(model, iterator, dim_reducer):
    model.eval()
    all_x_2d = []
    all_preds = []
    all_y = []
    temperature = lt.Tensor(np.array(1.0), requires_grad=False)
    with lt.no_grad():
        for x, y in iterator:
            logits = model(x)
            preds = F.softmax(logits / temperature)
            all_x_2d.append(dim_reducer.transform(x.numpy()))
            all_preds.append(preds.numpy())
            all_y.append(y.numpy())

    if len(all_x_2d) > 0:
        all_x_2d = np.concatenate(all_x_2d, axis=0)
        all_preds = np.concatenate(all_preds, axis=0)
        all_y = np.concatenate(all_y, axis=0)

    # Use softmax probability of class 1 as the predicted label for coloring
    positive_probs = all_preds[:, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    cmap = 'PiYG'

    sc1 = ax1.scatter(all_x_2d[:, 0], all_x_2d[:, 1], c=positive_probs, cmap=cmap, alpha=0.7)
    ax1.set_title('Positive Class Probability')

    sc2 = ax2.scatter(all_x_2d[:, 0], all_x_2d[:, 1], c=all_y, cmap=cmap, alpha=0.7)
    ax2.set_title('Real Class')

    fig.colorbar(sc1, label='Class Label', ax=(ax1, ax2), orientation='horizontal')
    plt.show()

In [ ]:
from sklearn.decomposition import PCA
pca_reducer = PCA(n_components=2)
print(len(std_dataset_train.X), len(std_dataset_test.X))
whole_dataset = datasets.merge_datasets([std_dataset_train, std_dataset_test])
print(len(whole_dataset.X))

pca_reducer.fit(np.asarray(whole_dataset.X))
dataset_iterator = datasets.DataLoader(std_dataset_test, batch_size=batch_size, shuffle=False, rng=rng)
plot_features(model, dataset_iterator, pca_reducer)

## SHAP

### Computation

In [ ]:
import shap

def predict_fn(X):
    model.eval()
    with lt.no_grad():
        X = lt.Tensor(X, requires_grad=False)
        logits = model(X)
        probs = F.softmax(logits)
    return probs.numpy()[:, 1]  # Return probability of class 1


background = np.asarray(std_dataset_train.X)
explainer = shap.Explainer(predict_fn, background, seed=seed)
shap_values = explainer(np.asarray(std_dataset_test.X))
feature_names = [(name[:-1] + ' ' + name[-1]).replace('_', ' ').title() for name in metadata['feature_names']]
shap_values.feature_names = feature_names
shap_values.data = np.asarray(dataset_test.X)

### Global Importance

In [ ]:
shap.summary_plot(shap_values, np.asarray(std_dataset_test.X))

### Top feature analysis

In [ ]:
top_features = np.argsort(np.abs(shap_values.values).mean(0))[-5:]  # Get indices of top 5 features
for feature in top_features:
    shap.plots.scatter(
        shap_values[:, feature],
        color=shap_values
    )

### Feature importance for specific instances

In [ ]:
# TODO: choose examples strategically, e.g. one true positive and one false negative
shap.plots.waterfall(shap_values[0])
shap.plots.waterfall(shap_values[1]) 